In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


class SalesDataAnalyzer:

    def __init__(self, file_path=None):
        self.data = None
        self.last_fig = None
        if file_path:
            self.load_data(file_path)

    def __del__(self):
        pass

    def load_data(self, file_path):
        try:
            self.data = pd.read_csv(file_path)
            print("Dataset loaded successfully!")
        except FileNotFoundError:
            print(f"Error: File not found at '{file_path}'.")
        except pd.errors.EmptyDataError:
            print("Error: The file is empty.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    def _check_data(self):
        if self.data==None:
            print("No dataset loaded. Please load a dataset first (Option 1).")
            return False
        return True

    def explore_data(self):
        if not self._check_data():
            return
        while True:
            print("="*85)
            print("\nExplore Data")
            print("1. Display the first 5 rows")
            print("2. Display the last 5 rows")
            print("3. Display column names")
            print("4. Display data types")
            print("5. Display basic info")
            print("6. Back to Main Menu")
            choice = input("\nEnter your choice: ").strip()
            match choice:
                case "1":
                    print(self.data.head())
                case "2":
                    print(self.data.tail())
                case "3":
                    print(list(self.data.columns))
                case "4":
                    print(self.data.dtypes)
                case "5":
                    self.data.info()
                    print(self.data.describe(include="all"))
                case "6":
                    print()
                    print("Back to Main Menu".center(80))
                    print("="*85)
                    break
                case _:
                    print("Invalid choice. Please try again.")

    def clean_data(self):
        if not self._check_data():
            return
        while True:
            print("="*85)
            print("\nHandle Missing Data")
            print("1. Display rows with missing values")
            print("2. Fill missing values with mean")
            print("3. Drop rows with missing values")
            print("4. Replace missing values with a specific value")
            print("5. Back to Main Menu")
            choice = input("\nEnter your choice: ").strip()
            match choice:
                case "1":
                    missing_rows = self.data[self.data.isnull().any(axis=1)]
                    if missing_rows.empty:
                        print("No missing values found in the dataset!")
                    else:
                        print(missing_rows)
                case "2":
                    numeric_cols = self.data.select_dtypes(include=[np.number]).columns
                    self.data[numeric_cols] = self.data[numeric_cols].fillna(
                        self.data[numeric_cols].mean()
                    )
                    print("Missing numeric values filled with column mean.")
                case "3":
                    before = len(self.data)
                    self.data = self.data.dropna()
                    after = len(self.data)
                    print(f"Dropped {before - after} row(s) with missing values.")
                case "4":
                    value = input("Enter the value to replace missing data with: ").strip()
                    self.data = self.data.fillna(value)
                    print(f"Missing values replaced with '{value}'.")
                case "5":
                    print()
                    print("Back to Main Menu".center(80))
                    print("="*85)
                    break
                case _:
                    print("Invalid choice. Please try again.")

    def mathematical_operations(self):
        if not self._check_data():
            return
        numeric_cols = list(self.data.select_dtypes(include=[np.number]).columns)
        if not numeric_cols:
            print("No numeric columns available for this operation.")
            return
        print(f"Available numeric columns: {numeric_cols}")
        col = input("Enter a numeric column name to convert to a Numpy array: ").strip()
        if col not in numeric_cols:
            print("Invalid column name.")
            return
        arr = self.data[col].to_numpy()
        print(f"Numpy array (first 10 values): {arr[:10]}")
        print(f"Indexing (arr[0]): {arr[0]}")
        print(f"Slicing (arr[2:6]): {arr[2:6]}")
        print(f"Array + 10 (element-wise): {(arr[:10] + 10)}")
        print(f"Array * 2 (element-wise): {(arr[:10] * 2)}")
        print(f"Square root of array: {np.sqrt(np.abs(arr[:10]))}")

    def combine_data(self, other_dataframe):
        if not self._check_data():
            return
        method = input("Combine using 'concat', 'merge', or 'join': ").strip().lower()
        match method:
            case "concat":
                self.data = pd.concat([self.data, other_dataframe], ignore_index=True)
                print("DataFrames combined using concat.")
            case "merge":
                key = input("Enter the common column name to merge on: ").strip()
                self.data = pd.merge(self.data, other_dataframe, on=key, how="outer")
                print("DataFrames combined using merge.")
            case "join":
                self.data = self.data.join(
                    other_dataframe.set_index(other_dataframe.columns[0]),
                    rsuffix="_other"
                )
                print("DataFrames combined using join.")
            case _:
                print("Invalid method selected.")

    def split_data(self):
        if not self._check_data():
            return
        print(f"Available columns: {list(self.data.columns)}")
        col = input("Enter a column name to split the data by (e.g., Region, Product): ").strip()
        if col not in self.data.columns:
            print("Invalid column name.")
            return
        groups = {value: sub_df for value, sub_df in self.data.groupby(col)}
        print(f"Data split into {len(groups)} DataFrame(s) based on '{col}':")
        for value, sub_df in groups.items():
            print(f"\n-- {col} = {value} ({len(sub_df)} rows) --")
            print(sub_df.head())
        return groups

    def search_sort_filter(self):
        if not self._check_data():
            return
        while True:
            print("="*85)
            print("\nSearch, Sort, Filter")
            print("1. Search for specific value in a column")
            print("2. Sort data by a column")
            print("3. Filter data by condition")
            print("4. Back to Main Menu")
            choice = input("\nEnter your choice: ").strip()
            match choice:
                case "1":
                    col = input("Enter column name to search in: ").strip()
                    if col not in self.data.columns:
                        print("Invalid column name.")
                        continue
                    value = input("Enter value to search for: ").strip()
                    result = self.data[self.data[col].astype(str) == value]
                    print(result if not result.empty else "No matching records found.")
                case "2":
                    col = input("Enter column name to sort by: ").strip()
                    if col not in self.data.columns:
                        print("Invalid column name.")
                        continue
                    order = input("Sort order ('asc' or 'desc'): ").strip().lower()
                    ascending = order != "desc"
                    print(self.data.sort_values(by=col, ascending=ascending))
                case "3":
                    col = input("Enter column name to filter on: ").strip()
                    if col not in self.data.columns:
                        print("Invalid column name.")
                        continue
                    op = input("Enter operator ('==', '>', '<', '>=', '<='): ").strip()
                    value = input("Enter value to compare against: ").strip()
                    try:
                        value_num = float(value)
                    except ValueError:
                        value_num = value
                    match op:
                        case "==":
                            filtered = self.data[self.data[col] == value_num]
                        case ">":
                            filtered = self.data[self.data[col] > value_num]
                        case "<":
                            filtered = self.data[self.data[col] < value_num]
                        case ">=":
                            filtered = self.data[self.data[col] >= value_num]
                        case "<=":
                            filtered = self.data[self.data[col] <= value_num]
                        case _:
                            print("Invalid operator.")
                            continue
                    print(filtered if not filtered.empty else "No matching records found.")
                case "4":
                    print()
                    print("Back to Main Menu".center(80))
                    print("="*85)
                    break
                case _:
                    print("Invalid choice. Please try again.")

    def aggregate_functions(self):
        if not self._check_data():
            return
        numeric_cols = list(self.data.select_dtypes(include=[np.number]).columns)
        if not numeric_cols:
            print("No numeric columns available.")
            return
        print(f"Available numeric columns: {numeric_cols}")
        col = input("Enter a numeric column to aggregate: ").strip()
        if col not in numeric_cols:
            print("Invalid column name.")
            return
        print(f"Sum: {self.data[col].sum()}")
        print(f"Mean: {self.data[col].mean()}")
        print(f"Count: {self.data[col].count()}")
        print(f"Min: {self.data[col].min()}")
        print(f"Max: {self.data[col].max()}")

    def statistical_analysis(self):
        if not self._check_data():
            return
        numeric_cols = list(self.data.select_dtypes(include=[np.number]).columns)
        if not numeric_cols:
            print("No numeric columns available.")
            return
        print(self.data[numeric_cols].describe())
        col = input("Enter a numeric column for detailed statistics: ").strip()
        if col not in numeric_cols:
            print("Invalid column name.")
            return
        print(f"Standard Deviation: {self.data[col].std()}")
        print(f"Variance: {self.data[col].var()}")
        print(f"25th Percentile: {self.data[col].quantile(0.25)}")
        print(f"50th Percentile: {self.data[col].quantile(0.50)}")
        print(f"75th Percentile: {self.data[col].quantile(0.75)}")

    def create_pivot_table(self):
        if not self._check_data():
            return
        print(f"Available columns: {list(self.data.columns)}")
        index_col = input("Enter column name for pivot table index: ").strip()
        values_col = input("Enter numeric column name for pivot table values: ").strip()
        agg_func = input("Enter aggregation function ('sum', 'mean', 'count'): ").strip().lower()
        if index_col not in self.data.columns or values_col not in self.data.columns:
            print("Invalid column name(s).")
            return
        if agg_func not in ("sum", "mean", "count"):
            print("Invalid aggregation function.")
            return
        pivot = pd.pivot_table(self.data, index=index_col, values=values_col, aggfunc=agg_func)
        print(pivot)

    def groupby_transform(self):
        if not self._check_data():
            return
        group_col = input("Enter column name to group by: ").strip()
        value_col = input("Enter numeric column name to transform: ").strip()
        if group_col not in self.data.columns or value_col not in self.data.columns:
            print("Invalid column name(s).")
            return
        grouped = self.data.groupby(group_col)[value_col].transform("mean")
        self.data[f"{value_col}_group_mean"] = grouped
        print(f"New column '{value_col}_group_mean' added using groupby/transform.")
        print(self.data.head())

    def reindex_data(self):
        if not self._check_data():
            return
        self.data = self.data.reset_index(drop=True)
        self.data.index = self.data.index + 1
        self.data.index.name = "S.No"
        print("Data re-indexed successfully.")
        print(self.data.head())

    def visualize_data(self):
        if not self._check_data():
            return
        numeric_cols = list(self.data.select_dtypes(include=[np.number]).columns)
        while True:
            print("="*85)
            print("\nData Visualization")
            print("1. Bar Plot")
            print("2. Line Plot")
            print("3. Scatter Plot")
            print("4. Pie Chart")
            print("5. Histogram")
            print("6. Stack Plot")
            print("7. Subplots (multiple plots)")
            print("8. Seaborn Heatmap")
            print("9. Seaborn Box Plot")
            print("10. Back to Main Menu")
            choice = input("\nEnter your choice: ").strip()
            match choice:
                case "1":
                    x_col = input("Enter column name for x-axis (categorical): ").strip()
                    y_col = input("Enter numeric column name for y-axis: ").strip()
                    grouped = self.data.groupby(x_col)[y_col].sum()
                    plt.figure()
                    plt.bar(grouped.index.astype(str), grouped.values)
                    plt.title(f"{y_col} by {x_col}")
                    plt.xlabel(x_col)
                    plt.ylabel(y_col)
                    plt.legend([y_col])
                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "2":
                    y_col = input("Enter numeric column name to plot over the row index: ").strip()
                    plt.figure()
                    plt.plot(self.data.index, self.data[y_col])
                    plt.title(f"{y_col} over records")
                    plt.xlabel("Index")
                    plt.ylabel(y_col)
                    plt.legend([y_col])
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "3":
                    x_col = input("Enter numeric column name for x-axis: ").strip()
                    y_col = input("Enter numeric column name for y-axis: ").strip()
                    plt.figure()
                    plt.scatter(self.data[x_col], self.data[y_col])
                    plt.title(f"{y_col} vs {x_col}")
                    plt.xlabel(x_col)
                    plt.ylabel(y_col)
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "4":
                    cat_col = input("Enter categorical column name: ").strip()
                    val_col = input("Enter numeric column name for values: ").strip()
                    grouped = self.data.groupby(cat_col)[val_col].sum()
                    plt.figure()
                    plt.pie(grouped.values, labels=grouped.index.astype(str), autopct="%1.1f%%")
                    plt.title(f"{val_col} share by {cat_col}")
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "5":
                    col = input("Enter numeric column name for histogram: ").strip()
                    plt.figure()
                    plt.hist(self.data[col].dropna(), bins=10)
                    plt.title(f"Distribution of {col}")
                    plt.xlabel(col)
                    plt.ylabel("Frequency")
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "6":
                    cat_col = input("Enter categorical column name (e.g., Region): ").strip()
                    val_col = input("Enter numeric column name (e.g., Sales): ").strip()
                    pivoted = self.data.pivot_table(index=self.data.index, columns=cat_col, values=val_col, aggfunc="sum").fillna(0)
                    plt.figure()
                    plt.stackplot(pivoted.index, [pivoted[c] for c in pivoted.columns], labels=pivoted.columns.astype(str))
                    plt.title(f"Stack Plot of {val_col} by {cat_col}")
                    plt.legend(loc="upper left")
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "7":
                    if len(numeric_cols) < 2:
                        print("Need at least two numeric columns for subplots.")
                        continue
                    plt.figure(figsize=(10, 8))
                    plt.subplot(2, 2, 1)
                    plt.hist(self.data[numeric_cols[0]].dropna(), bins=10)
                    plt.title(f"Histogram of {numeric_cols[0]}")
                    plt.subplot(2, 2, 2)
                    plt.plot(self.data.index, self.data[numeric_cols[0]])
                    plt.title(f"Line Plot of {numeric_cols[0]}")
                    plt.subplot(2, 2, 3)
                    plt.scatter(self.data[numeric_cols[0]], self.data[numeric_cols[-1]])
                    plt.title(f"{numeric_cols[0]} vs {numeric_cols[-1]}")
                    plt.subplot(2, 2, 4)
                    plt.boxplot(self.data[numeric_cols].dropna())
                    plt.title("Box Plot of Numeric Columns")
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "8":
                    if len(numeric_cols) < 2:
                        print("Need at least two numeric columns for a heatmap.")
                        continue
                    plt.figure()
                    sns.heatmap(self.data[numeric_cols].corr(), annot=True, cmap="coolwarm")
                    plt.title("Correlation Heatmap")
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "9":
                    col = input("Enter numeric column name for box plot: ").strip()
                    cat_col = input("Enter a categorical column to group by (or leave blank): ").strip()
                    plt.figure()
                    if cat_col and cat_col in self.data.columns:
                        sns.boxplot(x=cat_col, y=col, data=self.data)
                    else:
                        sns.boxplot(y=col, data=self.data)
                    plt.title(f"Box Plot of {col}")
                    plt.tight_layout()
                    plt.show()
                    self.last_fig = plt.gcf()
                case "10":
                    print()
                    print("Back to Main Menu".center(80))
                    print("="*85)
                    break
                case _:
                    print("Invalid choice. Please try again.")

    def save_visualization(self, filename):
        if self.last_fig == None:
            print("No visualization available to save. Please create one first (Option 6).")
            return
        try:
            self.last_fig.savefig(filename)
            print(f"Visualization saved as {filename} successfully!")
        except Exception as e:
            print(f"Error saving visualization: {e}")


def dataframe_operations_menu(analyzer):
    while True:
        print("="*85)
        print("\nPerform DataFrame Operations")
        print("1. Numpy Array Creation, Indexing & Slicing")
        print("2. Mathematical Operations")
        print("3. Combine DataFrames")
        print("4. Split DataFrame")
        print("5. Search, Sort, Filter")
        print("6. Aggregate Functions")
        print("7. Create Pivot Table")
        print("8. Groupby & Transform")
        print("9. Reindex Data")
        print("10. Back to Main Menu")
        choice = input("\nEnter your choice: ").strip()
        match choice:
            case "1":
                analyzer.mathematical_operations()
            case "2":
                analyzer.mathematical_operations()
            case "3":
                file_path = input("Enter path of the CSV file to combine with: ").strip()
                try:
                    other_df = pd.read_csv(file_path)
                    analyzer.combine_data(other_df)
                except Exception as e:
                    print(f"Error reading file: {e}")
            case "4":
                analyzer.split_data()
            case "5":
                analyzer.search_sort_filter()
            case "6":
                analyzer.aggregate_functions()
            case "7":
                analyzer.create_pivot_table()
            case "8":
                analyzer.groupby_transform()
            case "9":
                analyzer.reindex_data()
            case "10":
                print()
                print("Back to Main Menu".center(80))
                print("="*85)
                break
            case _:
                print("Invalid choice. Please try again.")


def main():
    analyzer = SalesDataAnalyzer()
    while True:
        print("="*85)
        print()
        print("Data Analysis & Visualization Program".center(85))
        print("\nPlease select an option:")
        print("1. Load Dataset")
        print("2. Explore Data")
        print("3. Perform DataFrame Operations")
        print("4. Handle Missing Data")
        print("5. Generate Descriptive Statistics")
        print("6. Data Visualization")
        print("7. Save Visualization")
        print("8. Exit")
        choice = input("\nEnter your choice: ").strip()
        match choice:
            case "1":
                print("="*85)
                print("\nLoad Dataset")
                file_path = input("\nEnter the path of the dataset (CSV file): ").strip()
                analyzer.load_data(file_path)
            case "2":
                analyzer.explore_data()
            case "3":
                dataframe_operations_menu(analyzer)
            case "4":
                analyzer.clean_data()
            case "5":
                analyzer.statistical_analysis()
            case "6":
                analyzer.visualize_data()
            case "7":
                print("="*85)
                print("\nSave Visualization")
                filename = input("\nEnter file name to save the plot (e.g., scatter_plot.png): ").strip()
                analyzer.save_visualization(filename)
            case "8":
                print()
                print("Exiting the program. Goodbye!".center(85))
                print("="*85)
                break
            case _:
                print("Invalid choice. Please try again.")


main()
